In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
sys.path.append(str(Path("..")))
from src.dataset import train_data, train_loader, val_loader, test_loader,test_path
from torchvision import transforms,datasets
from torch.utils.data import Subset,DataLoader

torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([1, 5, 0, 5, 3, 1, 7, 1, 1, 5])


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


# *DropOut 0.3*

In [3]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x


num_classes = len(train_data.classes)

model = CNN(num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []

for epoch in range(num_epochs):


    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = train_correct / train_total

    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average= None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_f1_scores.append(val_f1)
    val_f1_scores_by_class .append(f1_score_by_class)



    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )
    
    print('F1_Score by each class')
    print(f1_score_by_class)
    
    
    print('=======================================================================================')

Epoch [1/100] | Train Loss: 2.0150 | Train Acc: 0.3224 | Val Loss: 1.3279 | Val Acc: 0.5120 | Val Macro F1: 0.4469
F1_Score by each class
[0.52777778 0.59925094 0.26470588 0.4587156  0.56910569 0.58394161
 0.57142857 0.        ]
Epoch [2/100] | Train Loss: 1.0955 | Train Acc: 0.6096 | Val Loss: 0.8338 | Val Acc: 0.7192 | Val Macro F1: 0.7011
F1_Score by each class
[0.70503597 0.84916201 0.62727273 0.56050955 0.69072165 0.77981651
 0.85082873 0.54545455]
Epoch [3/100] | Train Loss: 0.7256 | Train Acc: 0.7451 | Val Loss: 0.8289 | Val Acc: 0.7057 | Val Macro F1: 0.6722
F1_Score by each class
[0.68027211 0.80555556 0.62135922 0.53030303 0.74418605 0.72340426
 0.80701754 0.46511628]
Epoch [4/100] | Train Loss: 0.5289 | Train Acc: 0.8078 | Val Loss: 0.6842 | Val Acc: 0.7838 | Val Macro F1: 0.7610
F1_Score by each class
[0.75177305 0.88205128 0.70351759 0.65882353 0.77777778 0.83962264
 0.90322581 0.57142857]
Epoch [5/100] | Train Loss: 0.3994 | Train Acc: 0.8544 | Val Loss: 0.6523 | Val Acc:

In [4]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)


model.eval()

all_test_labels = []
all_test_predictions = []

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        
test_accuracy = test_correct / test_total
test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)
test_f1_by_class = f1_score(
    all_test_labels,
    all_test_predictions,
    average=None
)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

print("Test F1 by class:")
print(test_f1_by_class)


Test Accuracy: 0.7372
Test Macro F1: 0.7286
Test F1 by class:
[0.69491525 0.92631579 0.57777778 0.54545455 0.78181818 0.78504673
 0.93069307 0.58666667]


# *DropOut 0.5*

In [5]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x


num_classes = len(train_data.classes)

model = CNN(num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []

for epoch in range(num_epochs):


    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = train_correct / train_total

    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average= None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_f1_scores.append(val_f1)
    val_f1_scores_by_class .append(f1_score_by_class)



    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )
    
    print('F1_Score by each class')
    print(f1_score_by_class)
    
    
    print('=======================================================================================')

Epoch [1/100] | Train Loss: 1.5486 | Train Acc: 0.4610 | Val Loss: 0.9801 | Val Acc: 0.6471 | Val Macro F1: 0.5814
F1_Score by each class
[0.61666667 0.69456067 0.37956204 0.54639175 0.65789474 0.71311475
 0.87619048 0.16666667]
Epoch [2/100] | Train Loss: 0.8161 | Train Acc: 0.7128 | Val Loss: 0.6786 | Val Acc: 0.7763 | Val Macro F1: 0.7604
F1_Score by each class
[0.76595745 0.88       0.67724868 0.66292135 0.7875     0.80334728
 0.88135593 0.625     ]
Epoch [3/100] | Train Loss: 0.5232 | Train Acc: 0.8093 | Val Loss: 0.5981 | Val Acc: 0.8033 | Val Macro F1: 0.7801
F1_Score by each class
[0.8        0.90052356 0.7032967  0.67045455 0.80239521 0.85321101
 0.91866029 0.59259259]
Epoch [4/100] | Train Loss: 0.3459 | Train Acc: 0.8836 | Val Loss: 0.4985 | Val Acc: 0.8318 | Val Macro F1: 0.8183
F1_Score by each class
[0.81578947 0.9119171  0.74468085 0.70157068 0.81818182 0.88995215
 0.95876289 0.70588235]
Epoch [5/100] | Train Loss: 0.2324 | Train Acc: 0.9182 | Val Loss: 0.5453 | Val Acc:

In [6]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)


model.eval()

all_test_labels = []
all_test_predictions = []

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        
test_accuracy = test_correct / test_total
test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)
test_f1_by_class = f1_score(
    all_test_labels,
    all_test_predictions,
    average=None
)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

print("Test F1 by class:")
print(test_f1_by_class)


Test Accuracy: 0.7704
Test Macro F1: 0.7641
Test F1 by class:
[0.83870968 0.88421053 0.66101695 0.66055046 0.84536082 0.80373832
 0.94845361 0.47058824]
